In [0]:
%pip install transformers
dbutils.library.restartPython()

In [0]:
#Imports & config
import hashlib
from typing import List
from transformers import AutoTokenizer

CATALOG = "rag_based_chatbot"
SCHEMA = "default"
TABLE_NAME = f"{CATALOG}.{SCHEMA}.rag_documents"   # SAME table your Vector Search
                                                     # index already points at — this
                                                     # script REPLACES the word-based
                                                     # chunker that filled it before,
                                                     # it doesn't create a new pipeline

SOURCE_TABLE = f"{CATALOG}.{SCHEMA}.silver_reviews_with_emoji"

# WHY this tokenizer, not a generic one: it should match whichever model's
# token limit you're actually chunking FOR. If you're chunking for the
# embedding step (databricks-bge-large-en, ~512 token context), tokenize
# with that model's tokenizer, not the LLM's — a Llama tokenizer and a
# BGE tokenizer don't split text into tokens the same way, so "500 tokens"
# by one count isn't "500 tokens" by the other. Using the LLM's tokenizer
# here (as your chain.py already assumes) is fine AS LONG AS your embedding
# model's real limit is checked separately (see the note after Cell 4).
TOKENIZER_MODEL = "llama_v3_3_70b_instruct"

# WHY 400, not 500 like the PDF template: your document_text wraps the
# chunk in "Product: {asin}. Rating: {overall}/5. Summary: {summary}.
# Review: {chunk}" — that wrapper costs tokens too. Budgeting the review
# body at 400 leaves ~100 tokens of headroom for the wrapper before
# hitting a 512-token embedding ceiling. The PDF template didn't need
# this margin because it embedded raw chunk_text with no wrapper.
MAX_TOKENS_PER_CHUNK = 400

# WHY these separators, not the PDF template's ["\n\n","\n"," ",""]:
# reviews are short, single-paragraph prose with no newline structure —
# "\n\n"/"\n" would almost never fire and just waste a pass. Splitting
# on sentence boundaries first (". ", "! ", "? ") keeps a chunk's meaning
# intact far better than a mid-sentence word cut; " " and "" remain as
# fallbacks for the rare run-on review with no punctuation at all.
SEPARATORS = [". ", "! ", "? ", " ", ""]

In [0]:
def hash_id(raw_id: str) -> str:
    return hashlib.sha256(str(raw_id).encode()).hexdigest()[:16]

In [0]:
def chunk_text(text: str, tokenizer: AutoTokenizer) -> List[str]:
    """Splits text into chunks under MAX_TOKENS_PER_CHUNK by progressively
    trying finer separators (sentence -> word -> character)."""
    chunks = [text]
    for separator in SEPARATORS:
        new_chunks = []
        for chunk in chunks:
            if len(tokenizer.encode(chunk)) > MAX_TOKENS_PER_CHUNK:
                parts = chunk.split(separator)
                current_chunk = ""
                for part in parts:
                    combined = current_chunk + separator + part if current_chunk else part
                    if len(tokenizer.encode(combined)) <= MAX_TOKENS_PER_CHUNK:
                        current_chunk = combined
                    else:
                        if current_chunk:
                            new_chunks.append(current_chunk.strip())
                        # part itself still too large -> hard token slice
                        while len(tokenizer.encode(part)) > MAX_TOKENS_PER_CHUNK:
                            tokens = tokenizer.encode(part)
                            sub_part = tokenizer.decode(tokens[:MAX_TOKENS_PER_CHUNK])
                            new_chunks.append(sub_part.strip())
                            part = tokenizer.decode(tokens[MAX_TOKENS_PER_CHUNK:])
                        current_chunk = part.strip()
                if current_chunk:
                    new_chunks.append(current_chunk.strip())
            else:
                new_chunks.append(chunk.strip())
        chunks = new_chunks

    # Final safety pass: guarantee no chunk exceeds the limit, even after
    # falling through every separator (mirrors the PDF template exactly —
    # this part didn't need to change, it's already source-agnostic)
    final_chunks = []
    for chunk in chunks:
        tokens = tokenizer.encode(chunk)
        start = 0
        while start < len(tokens):
            sub_tokens = tokens[start:start + MAX_TOKENS_PER_CHUNK]
            final_chunks.append(tokenizer.decode(sub_tokens).strip())
            start += MAX_TOKENS_PER_CHUNK

    return [c for c in final_chunks if c]  # drop any empty strings


In [0]:
def process_review(row: dict, tokenizer: AutoTokenizer) -> List[dict]:
    """Chunks one review and returns one dict per chunk, ready to insert."""
    if not isinstance(row["reviewText"], str) or len(row["reviewText"].split()) < 8:
        return []  # same low-signal filter as your original chunker

    reviewer_hash = hash_id(row["reviewerID"])
    chunks = chunk_text(row["reviewText"], tokenizer)

    records = []
    for i, chunk in enumerate(chunks):
        doc_text = (
            f"Product: {row['asin']}. Rating: {row['overall']}/5. "
            f"Summary: {row['summary_clean']}. Review: {chunk}"
        )
        records.append({
            "doc_id": f"{reviewer_hash}_{row['asin']}_{i}",
            "reviewer_hash": reviewer_hash,
            "asin": row["asin"],
            "overall": int(row["overall"]),
            "chunk_index": i,
            "n_chunks": len(chunks),
            "document_text": doc_text,
            # carried straight from Silver, same as the previous step —
            # sentiment is computed once per review, not per chunk
            "summary_emoji": row["summary_emoji"],
            "summary_sentiment_net": int(row["summary_sentiment_net"]),
        })
    return records

In [0]:
print(f"Loading tokenizer '{TOKENIZER_MODEL}'...")
tokenizer = AutoTokenizer.from_pretrained("hf-internal-testing/llama-tokenizer", cache_dir="/tmp/hf_cache")

print(f"Checking for table '{TABLE_NAME}'...")
try:
    spark.sql(f"DESCRIBE TABLE {TABLE_NAME}")
    print(f"Table '{TABLE_NAME}' already exists. Truncating previous chunks...")
    spark.sql(f"TRUNCATE TABLE {TABLE_NAME}")
    spark.sql(f"ALTER TABLE {TABLE_NAME} SET TBLPROPERTIES (delta.enableChangeDataFeed = true)")
except Exception:
    print(f"Table '{TABLE_NAME}' not found. Creating it...")
    spark.sql(f"""
        CREATE TABLE {TABLE_NAME} (
            doc_id STRING,
            reviewer_hash STRING,
            asin STRING,
            overall INT,
            chunk_index INT,
            n_chunks INT,
            document_text STRING,
            summary_emoji STRING,
            summary_sentiment_net INT
        )
        TBLPROPERTIES (delta.enableChangeDataFeed = true)
    """)
print(f"Table '{TABLE_NAME}' ready for new data.")

# WHY read as pandas via toPandas() here rather than a pure-Spark UDF:
# AutoTokenizer isn't trivially serializable to Spark executors without
# extra setup (broadcast + pandas_udf), and at ~57k rows this fits
# comfortably in driver memory — same "don't reach for distributed
# tooling below the point it pays off" reasoning used earlier in this
# project's pandas-vs-PySpark decision.
import pandas as pd

silver_pdf = pd.read_pickle(f"/Volumes/rag_based_chatbot/default/documents/silver_reviews_with_emoji.pkl")

print(f"Chunking {len(silver_pdf)} reviews...")
all_records = []
for _, row in silver_pdf.iterrows():
    all_records.extend(process_review(row.to_dict(), tokenizer))

print(f"Produced {len(all_records)} chunks from {len(silver_pdf)} reviews.")

# Single batch write — unlike the PDF template's per-file append (which
# made sense when files arrived one at a time), all reviews are already
# in memory together, so one write is simpler and avoids 57k tiny appends
spark.createDataFrame(all_records).write.format("delta").mode("append").saveAsTable(TABLE_NAME)

n_split = len({r["doc_id"].rsplit("_", 1)[0] for r in all_records if r["n_chunks"] > 1})
print(f"[rag_prep] {len(silver_pdf)} reviews -> {len(all_records)} chunks "
      f"({n_split} reviews split into multiple chunks)")
print("All reviews processed and inserted.")